In [1]:
## Load in python Code :
from google.colab import userdata
openai_key = userdata.get('OPENAI')  ## For OpenAI Models

import os
os.environ['OPENAI_API_KEY'] = openai_key

In [2]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.5 MB/s eta 0:00:00


In [3]:
!pip install pypdf langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 238.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 250.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 205.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 220.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 294.7 kB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [5]:
# Document Loader :
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/attention-is-all-you-need-Paper.pdf")

pages = loader.load()

In [6]:
# Text Splitter :
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000, ## no of  characters
    chunk_overlap  = 200,
    )

chunks = text_splitter.split_documents(pages)

In [7]:
# Embedding Model :
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings()

In [4]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelem

In [8]:
# Vector DB :

from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory='mydb'
    )

###Retriever :

In [9]:
retriever = db.as_retriever()

In [11]:
len(retriever.invoke("what is attention mechanism?"))

4

In [13]:
## choosing no. of chunks to retrieve : default=4
retriever = db.as_retriever(search_kwargs={'k': 9})

len(retriever.invoke("what is attention mechanism?"))

9

In [ ]:
## Search_type : default="similarity"

"similarity"  : Chunks with max similarity will be retrieved

"mmr" (maximal marginal relevence) : --> diverse data
>> chunk & query similarity are calculated
>> calculate similarity of retrieved chunks with other
>> if similarity of chunk with ecah other is more than threshold, drop one of that chunk


"similarity_score_threshold" :
retrive chunks with similarity score greater than threshold

In [14]:
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={'k': 3}
    )

In [ ]:
1
2
3
4
5
6
7


1
---4
7
5
2

In [15]:
retriever.invoke("what is capital of India?")

[Document(metadata={'created': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'page_label': '11', 'eventtype': 'Poster', 'creator': 'PyPDF', 'source': '/content/attention-is-all-you-need-Paper.pdf', 'lastpage': '6008', 'moddate': '2018-02-12T21:22:10-08:00', 'title': 'Attention is All you Need', 'page': 10, 'type': 'Conference Proceedings', 'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. Fergus and S. Vishwanathan and R. Garnett', 'publisher': 'Curran Associates, Inc.', 'firstpage': '5998', 'language': 'en-US', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing wit

In [19]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={'k': 5, "score_threshold" : 0.7}
    )

len(retriever.invoke("what is attention?"))

4

###Multi-Query Retriever :

In [ ]:
"what is attention?"
"what is attention mechanism?"
"Tell me about attention"
"explain attention"

In [ ]:
## basic retriever :
query --> db --> embedding model --> similarity(chunk, query) --> relevent chunks


## Multi-query retriever :
query  --> LLM --> 3 queries --> db --> embedding model --> similarity(chunk, query) --> unique(relevent chunks)

In [ ]:
"based on given question create more similar question"

In [20]:
base_retriever = db.as_retriever()

## AI Model :
from langchain_openai import OpenAI

model = OpenAI()

In [21]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=model
    )

In [23]:
len(multi_retriever.invoke("what is attention?"))

6

###Parent Document Retriever :

In [ ]:
chunk = "attention is....."

chunk = """
attention is ...
jhvhjv
jvh
ukv
"""

In [ ]:
If chunk size is smaller --> retriever will work better
if chunk size is bigger --> Model will have better context to generate answer

In [ ]:
4k     --> 1k, 1k, 1k, 1k
4k     --> 1k, 1k, 1k, 1k
4k     --> 1k, 1k, 1k, 1k
4k     --> 1k, 1k, 1k, 1k
            (child chunks)
Parent
chunk

In [ ]:
## Working :
-- Parent splitter : will create parent chunks (large chunks)
-- Child splitter : split parent chunk into smaller chunks (child chunk)
-- For better retriever : we store child chunk in vectordb
-- but while passing chunks to llm, we pass parent chunk

In [24]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 4000,
    chunk_overlap  = 0,
    )

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 0,
    )

In [25]:
## create empty db : to store child chunk
new_db = Chroma(
    collection_name='mydbtr',
    embedding_function=embedding_model,
)

/tmp/ipykernel_6587/3998320033.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  new_db = Chroma(


In [26]:
## To store parent chunk :
from langchain_core.stores import InMemoryStore

parent_store = InMemoryStore()

In [27]:
from langchain_classic.retrievers import ParentDocumentRetriever

parent_retriever = ParentDocumentRetriever(
    vectorstore=new_db,    ## To store child chunk
    docstore=parent_store, ## To store parent chunk
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

parent_retriever.add_documents(pages)

In [29]:
parent_retriever.invoke("what is attention?")

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On 

###Hybrid/Ensemble retriver :

In [ ]:
# BM-25 --> keyword based retriver

In [ ]:
## Hybrid/Ensemble retriever :
-- BM25 + VectorDB
-- We are combining keyword based retriever & vector based retriever

In [32]:
!pip install rank_bm25

In [33]:
from langchain_classic.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(
    documents=chunks,
)

In [34]:
vector_retriever = db.as_retriever()

In [35]:
from langchain_classic.retrievers import EnsembleRetriever

In [36]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
)